# 02 — Temporal stroke NVU / PET target screen (GSE227651)

Four published **10x matrices**, mouse ipsilateral hemisphere after **tMCAO**:
- sham (day 7 postoperative control; GSM7104635)
- MCAO day 1 (GSM7104632)
- MCAO day 3 (GSM7104633)
- MCAO day 7 (GSM7104634)

**Critical study-design rule:** one deposited GEO count matrix per condition; the experiment does not establish independent animal-level replication. Here, cells are **not** used as replicates. There are **no inferential p values, adjusted p values, or biological-replicate CIs** in this notebook. Every trajectory is a descriptive discovery result requiring independent replication.

The gene list was frozen before viewing this dataset. Host targets are not advantaged. Trajectories do not establish causal ordering, tracer accessibility or drug delivery. <https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE227651>


In [ ]:
!pip -q install "scanpy>=1.10,<1.12" "anndata>=0.10" "leidenalg>=0.10" "igraph>=0.11"

import os, gzip, hashlib, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy.io import mmread
import scanpy as sc, anndata as ad, requests, matplotlib.pyplot as plt

ROOT=Path(os.environ.get("IMACS_ROOT", "/content/imacs_stroke_pet_temporal" if Path("/content").exists() else "imacs_stroke_pet_temporal"))
RAW=ROOT/"raw"; OUT=ROOT/"results"; FIG=ROOT/"figures"
for d in (RAW,OUT,FIG): d.mkdir(exist_ok=True,parents=True)
print(sys.version, sc.__version__,ROOT)


In [ ]:
CAT_URL="https://raw.githubusercontent.com/Touf00/retinal-nvu-computational-analysis/imacs-stroke-pet-screen/IMACS_Stroke_PET/config/candidate_catalog.csv"
CAT=pd.read_csv(CAT_URL).drop_duplicates("gene")
SAMPLES=pd.DataFrame([
 ("GSM7104635","sham","Sham"),
 ("GSM7104632","1d","MCAO_day1"),
 ("GSM7104633","3d","MCAO_day3"),
 ("GSM7104634","7d","MCAO_day7"),
],columns=["gsm","prefix","condition"])
display(SAMPLES,CAT)


## Download actual sample supplemental files and verify checksums

Each sample uses the exact GEO naming convention (`GSM7104632_1d-...`, etc.). The features are *features.tsv.gz*, not *genes.tsv.gz*.

If the GEO host is unavailable, stop rather than silently substituting another dataset.

In [ ]:
def download(url,p):
    p=Path(p)
    if p.is_file() and p.stat().st_size>0: return p
    with requests.get(url,stream=True,timeout=240) as r:
        r.raise_for_status()
        with p.open("wb") as f:
            for chunk in r.iter_content(chunk_size=1<<20):
                if chunk:f.write(chunk)
    return p

def digest(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""):h.update(chunk)
    return h.hexdigest()

objects=[]; provenance=[]
for gsm,prefix,condition in SAMPLES.itertuples(index=False):
    d=RAW/gsm;d.mkdir(exist_ok=True)
    urlbase=f"https://ftp.ncbi.nlm.nih.gov/geo/samples/{gsm[:-3]}nnn/{gsm}/suppl/"
    ps={}
    for suf in ("barcodes.tsv.gz","features.tsv.gz","matrix.mtx.gz"):
        p=download(urlbase+f"{gsm}_{prefix}-{suf}",d/f"{gsm}_{prefix}-{suf}")
        ps[suf]=p
        provenance.append({"gsm":gsm,"sample_condition":condition,"filename":p.name,
                           "sha256":digest(p),"bytes":p.stat().st_size})
    with gzip.open(ps["matrix.mtx.gz"],"rb") as f: x=mmread(f).T.tocsr()
    feat=pd.read_csv(ps["features.tsv.gz"],sep="\t",header=None,compression="gzip")
    genes=feat.iloc[:,1].astype(str).tolist() if feat.shape[1]>1 else feat.iloc[:,0].astype(str).tolist()
    with gzip.open(ps["barcodes.tsv.gz"],"rt") as f: barcodes=[z.rstrip() for z in f]
    a=ad.AnnData(x)
    a.var_names=pd.Index(genes); a.var_names_make_unique()
    a.obs_names=[f"{gsm}:{b}" for b in barcodes]
    a.obs["sample_id"]=gsm; a.obs["condition"]=condition
    objects.append(a)
    print(gsm,condition,a.shape)

pd.DataFrame(provenance).to_csv(OUT/"02_temporal_source_SHA256.csv",index=False)
a=ad.concat(objects,join="outer",merge="same")
assert a.obs_names.is_unique
print(a)


## QC and broad compartment recovery

The same broad compartments are used across all timepoints; all sample cell counts are reported. The schema explicitly labels ambiguous cluster identities and reports marker-score margins. Watch for clustering biased toward injury/immune composition.

In [ ]:
a.var["mt"]=a.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(a,qc_vars=["mt"],inplace=True)
qc=a.obs.groupby("condition")[["n_genes_by_counts","total_counts","pct_counts_mt"]].agg(["median","mean"])
qc.to_csv(OUT/"02_temporal_qc_before.csv")
display(qc)

keep=(a.obs["n_genes_by_counts"].to_numpy()>=200) & (a.obs["pct_counts_mt"].to_numpy()<20)
for sample,idx in a.obs.groupby("sample_id").groups.items():
    i=a.obs_names.get_indexer(idx)
    sub=a.obs.iloc[i]
    keep[i]&=sub["n_genes_by_counts"].to_numpy()<=sub["n_genes_by_counts"].quantile(.995)
    keep[i]&=sub["total_counts"].to_numpy()<=sub["total_counts"].quantile(.995)
a=a[keep].copy()
a.layers["counts"]=a.X.copy()
sc.pp.normalize_total(a,target_sum=1e4); sc.pp.log1p(a); a.raw=a.copy()
print(a.obs["condition"].value_counts().to_string())


In [ ]:
MARK={
"Endothelial":["Cldn5","Pecam1","Kdr","Esam","Emcn","Ramp2"],
"Pericyte":["Pdgfrb","Rgs5","Kcnj8","Abcc9","Anpep","Cd248"],
"VSMC":["Acta2","Tagln","Myh11","Cnn1"],
"Astrocyte":["Aqp4","Slc1a3","Aldoc","Gja1","Aldh1l1"],
"Microglia":["P2ry12","Tmem119","C1qa","Hexb"],
"Myeloid":["Ptprc","Lyz2","Tyrobp","Aif1","Fcgr3"],
"Neuron":["Rbfox3","Snap25","Syt1","Map2"],
"Oligodendrocyte":["Mbp","Plp1","Mog","Mag"],
"OPC":["Pdgfra","Olig1","Olig2"],
"VLMC_Fibroblast":["Col1a1","Col1a2","Dcn","Lum"],
"Ependymal_CP":["Ttr","Foxj1","Clic6"]
}
sc.pp.highly_variable_genes(a,n_top_genes=2500,flavor="seurat",batch_key="sample_id")
sc.tl.pca(a,n_comps=40,use_highly_variable=True)
sc.pp.neighbors(a,n_neighbors=15,n_pcs=30);sc.tl.umap(a)
sc.tl.leiden(a,resolution=.6,key_added="cluster")

for ct,genes in MARK.items():
    present=[g for g in genes if g in a.raw.var_names]
    if len(present)<2: raise RuntimeError(f"Too few lineage markers for {ct}: {present}")
    sc.tl.score_genes(a,gene_list=present,score_name=f"lineage_{ct}",use_raw=True)

cols=[f"lineage_{x}" for x in MARK]
ct=a.obs.groupby("cluster")[cols].mean()
ct["best"]=ct.idxmax(axis=1).str.replace("lineage_","",regex=False)
ranks=np.sort(ct[cols].to_numpy(),axis=1)
ct["margin"]=ranks[:,-1]-ranks[:,-2]
ct["cell_type"]=np.where(ct["margin"]>.05,ct["best"],"Ambiguous")
a.obs["broad_celltype"]=a.obs["cluster"].map(ct["cell_type"]).astype("category")
ct.to_csv(OUT/"02_temporal_cluster_annotations.csv")
counts=pd.crosstab(a.obs["condition"],a.obs["broad_celltype"])
counts.to_csv(OUT/"02_temporal_cell_counts.csv")
display(ct,counts)
sc.pl.umap(a,color=["condition","broad_celltype"],show=True)


## Descriptive candidate trajectories

Report **mean log1p counts per cell**, **detection fraction**, and **mean normalized per-cell CPM equivalent**, never p-values. Display means and detected fractions separately to avoid confusing increased expression with changing composition. Normalize per cell, stratify by cell type.

In [ ]:
def summaries(a,genelist):
    rows=[]; vmap={g:i for i,g in enumerate(a.var_names)}
    group=a.obs.groupby(["condition","broad_celltype"],observed=True).indices
    for g in genelist:
        if g not in vmap:
            rows.append({"gene":g,"condition":"UNDETECTED_GENE","broad_celltype":"NA","n_cells":0,
                         "mean_log1p":np.nan,"detection_fraction":np.nan,"mean_normalized":np.nan})
            continue
        j=vmap[g]
        for (condition,ct),idx in group.items():
            y=a.X[idx,j];y=np.asarray(y.toarray()).ravel() if sp.issparse(y) else np.asarray(y).ravel()
            z=a.layers["counts"][idx,j];z=np.asarray(z.toarray()).ravel() if sp.issparse(z) else np.asarray(z).ravel()
            rows.append({"gene":g,"condition":condition,"broad_celltype":ct,"n_cells":len(idx),
                         "mean_log1p":float(y.mean()),"detection_fraction":float((z>0).mean()),
                         "mean_normalized":float(np.expm1(y).mean())})
    return pd.DataFrame(rows)

T=summaries(a,CAT.gene.tolist())
T=T.merge(CAT,on="gene",how="left")
T.to_csv(OUT/"02_candidate_timecourse_descriptive.csv",index=False)
display(T.query('broad_celltype in ["Endothelial","Astrocyte","Pericyte"]').head(100))


In [ ]:
groups=["Endothelial","Astrocyte","Pericyte"]
for celltype in groups:
    sub=T[T.broad_celltype==celltype]
    if sub.empty:continue
    pv=sub.pivot_table(index="gene",columns="condition",values="mean_normalized")
    if "Sham" in pv.columns:
        for c in ["MCAO_day1","MCAO_day3","MCAO_day7"]:
            pv[f"log2ratio_{c}_vs_sham"]=np.log2((pv.get(c,np.nan)+0.1)/(pv["Sham"]+0.1))
    pv.to_csv(OUT/f"02_{celltype}_descriptive_ratios.csv")
    display(pv.head(30))


## Readout and decision limits

- These are **descriptive** trends from one sequenced sample per timepoint.
- No permutation tests over cells, no FDR, and no cell-based pseudo-confidence intervals.
- Aplnr/APJ and Amot/AMOT RNA abundance are not sufficient to establish accessible protein.
- BBB receptors can be present on abluminal rather than luminal membranes and do not prove productive transcytosis.
- The independent replicated 24 h experiment, spatial atlas and protein evidence must decide whether a lead survives.


In [ ]:
import json
(OUT/"02_run_manifest.json").write_text(json.dumps({
"accession":"GSE227651","conditions":SAMPLES.to_dict(orient="records"),
"statistical_inference":"NONE: one deposited sample per condition",
"candidate_count":len(CAT),
"analysis_version":"1.0"
},indent=2))
print("Saved", len(list(OUT.iterdir())),"files at", OUT)
